In [ ]:
!pip install catboost
!pip uninstall -y scikit-learn
!pip install scikit-learn==1.5.2
!pip uninstall -y xgboost
!pip install xgboost==2.1.3


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.7/98.7 MB 5.3 MB/s eta 0:00:00
Found existing installation: scikit-learn 1.6.1
Uninstalling scikit-learn-1.6.1:
  Successfully uninstalled scikit-learn-1.6.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 93.1 MB/s eta 0:00:00
Found existing installation: xgboost 2.1.4
Uninstalling xgboost-2.1.4:
  Successfully uninstalled xgboost-2.1.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.9/153.9 MB 6.2 MB/s eta 0:00:00


In [40]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
import pandas as pd
import numpy as np

from sklearn.ensemble import VotingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score

from imblearn.over_sampling import SMOTE
from sklearn.model_selection import KFold
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from xgboost.sklearn import XGBClassifier
import random

In [41]:
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

**데이터 전처리**

In [42]:
train = pd.read_csv('./train.csv').drop(columns={'ID','방수','욕실수','제공플랫폼','매물확인방식','방향','주차가능여부','해당층','총층','전용면적','총주차대수'})
test = pd.read_csv('./test.csv').drop(columns={'ID','방수','욕실수','제공플랫폼','매물확인방식','방향','주차가능여부','해당층','총층','전용면적','총주차대수'})

In [43]:
train['게재일'] = pd.to_datetime(train['게재일'])

train['days_since_first_post'] = (train['게재일'] - train['게재일'].min()).dt.days

train['day_of_year'] = train['게재일'].dt.dayofyear

# 변환
train['cos_day'] = np.cos(2 * np.pi * train['day_of_year'] / 365)
train['sin_day'] = np.sin(2 * np.pi * train['day_of_year'] / 365)

test['게재일'] = pd.to_datetime(test['게재일'])

test['days_since_first_post'] = (test['게재일'] - train['게재일'].min()).dt.days

test['day_of_year'] = test['게재일'].dt.dayofyear

# 변환
test['cos_day'] = np.cos(2 * np.pi * test['day_of_year'] / 365)
test['sin_day'] = np.sin(2 * np.pi * test['day_of_year'] / 365)

train['관리비'] *= 10000
train = train[train['관리비'] <= 200000]
train.reset_index(drop=True,inplace=True)

test['관리비'] *= 10000

train.drop(columns={'게재일','day_of_year'},inplace=True)
test.drop(columns={'게재일','day_of_year'},inplace=True)

<ipython-input-43-1fbc182957d3>:27: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train.drop(columns={'게재일','day_of_year'},inplace=True)


In [ ]:
train['중개사무소'].value_counts()

,count
중개사무소,
G52Iz8V2B9,799
r82ax9M3U3,43
J52gJ2E4T6,32
H90uE4C0W5,30
b87Td0W4Y3,27
...,...
m75Dz8P6I7,1
A21Yr4B1U8,1
g11ci7P5V1,1


**Frequency 인코딩**

In [44]:
agency_counts = train['중개사무소'].value_counts()

train['중개사무소_Count'] = train['중개사무소'].map(agency_counts)

test['중개사무소_Count'] = test['중개사무소'].apply(
    lambda x: agency_counts[x] if x in agency_counts else -1
)

train.drop(columns={'중개사무소'},inplace=True)
test.drop(columns={'중개사무소'},inplace=True)

In [45]:
X = train.drop(columns=['허위매물여부'])
y = train['허위매물여부']

In [46]:
X

,보증금,월세,관리비,days_since_first_post,cos_day,sin_day,중개사무소_Count
0,170500000.0,200000,0,608,0.997630,-0.068802,2
1,114000000.0,380000,0,580,0.852078,-0.523416,18
2,163500000.0,30000,100000,578,0.833556,-0.552435,797
3,346000000.0,530000,0,424,-0.995521,0.094537,19
4,153000000.0,530000,0,503,-0.300820,-0.953681,19
...,...,...,...,...,...,...,...
2430,159000000.0,550000,0,568,0.726608,-0.687053,16
2431,158500000.0,750000,20000,527,0.107381,-0.994218,11
2432,329000000.0,610000,100000,383,-0.696376,0.717677,797
2433,31000000.0,400000,80000,466,-0.809017,-0.587785,19


**하이퍼파라미터 튜닝**

In [ ]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 364.4/364.4 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 233.5/233.5 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.6/78.6 kB 5.9 MB/s eta 0:00:00


In [ ]:
import optuna

# 데이터 분리
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42,shuffle=True)

# SMOTE 적용
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

# XGBoost Optuna 최적화
def optimize_xgboost(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 820, 840),
        'max_depth': trial.suggest_int('max_depth', 11, 13),
        'learning_rate': trial.suggest_float('learning_rate', 0.2, 0.3),
        'subsample': trial.suggest_float('subsample', 0.5, 0.6),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 0.8),
        'min_child_weight': trial.suggest_float('min_child_weight', 5, 6),
        'gamma': trial.suggest_float('gamma', 0, 1),
        'random_state': 42,
        'eval_metric': 'logloss'
    }
    model = XGBClassifier(**params)
    model.fit(X_train_smote, y_train_smote)
    y_pred = model.predict(X_test)
    return f1_score(y_test, y_pred, average='macro')

study_xgb = optuna.create_study(direction='maximize')
study_xgb.optimize(optimize_xgboost, n_trials=100)
print("Best parameters for XGBoost:", study_xgb.best_params)
print("Best Macro F1 Score for XGBoost:", study_xgb.best_value)

[I 2025-01-16 03:55:18,382] A new study created in memory with name: no-name-f1f65263-0f1b-4c20-a370-835a9f746ce9
[I 2025-01-16 03:55:19,469] Trial 0 finished with value: 0.8631664615891409 and parameters: {'n_estimators': 835, 'max_depth': 13, 'learning_rate': 0.28799344304368124, 'subsample': 0.5669466681438123, 'colsample_bytree': 0.7354274334658338, 'min_child_weight': 5.543153652146846, 'gamma': 0.6208529842686266}. Best is trial 0 with value: 0.8631664615891409.
[I 2025-01-16 03:55:20,332] Trial 1 finished with value: 0.8825657101519171 and parameters: {'n_estimators': 834, 'max_depth': 12, 'learning_rate': 0.2670393103293681, 'subsample': 0.5014795473949711, 'colsample_bytree': 0.7018162584804811, 'min_child_weight': 5.983888061042634, 'gamma': 0.7070709877502478}. Best is trial 1 with value: 0.8825657101519171.
[I 2025-01-16 03:55:21,245] Trial 2 finished with value: 0.8785766288683441 and parameters: {'n_estimators': 822, 'max_depth': 13, 'learning_rate': 0.21976177964344826, 

Best parameters for XGBoost: {'n_estimators': 832, 'max_depth': 12, 'learning_rate': 0.20593358762047578, 'subsample': 0.5075069759848869, 'colsample_bytree': 0.7424805892464985, 'min_child_weight': 5.409454498304978, 'gamma': 0.9114298633986331}
Best Macro F1 Score for XGBoost: 0.9006527947776417


In [ ]:
# LightGBM Optuna 최적화
def optimize_lightgbm(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 345, 355),
        'max_depth': trial.suggest_int('max_depth', 65, 67),
        'learning_rate': trial.suggest_float('learning_rate', 0.15, 0.17),
        'subsample': trial.suggest_float('subsample', 0.5, 0.6),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 0.8),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 6),
        'num_leaves': trial.suggest_int('num_leaves', 27, 28),
        #'lambda_l1': trial.suggest_float('lambda_l1', 0.0, 10.0),
        #'lambda_l2': trial.suggest_float('lambda_l2', 0.0, 10.0),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.6, 0.7),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.7, 0.8),
        'random_state': 42,
        'verbose': -1
    }
    model = LGBMClassifier(**params)
    model.fit(X_train_smote, y_train_smote)
    y_pred = model.predict(X_test)
    return f1_score(y_test, y_pred, average='macro')

study_lgb = optuna.create_study(direction='maximize')
study_lgb.optimize(optimize_lightgbm, n_trials=100)
print("Best parameters for LightGBM:", study_lgb.best_params)
print("Best Macro F1 Score for LightGBM:", study_lgb.best_value)


[I 2025-01-16 03:52:52,237] A new study created in memory with name: no-name-3d3528d5-6323-4cfd-b951-fa5214fa6798
[I 2025-01-16 03:52:53,030] Trial 0 finished with value: 0.8705873402312843 and parameters: {'n_estimators': 354, 'max_depth': 67, 'learning_rate': 0.16334304772766467, 'subsample': 0.5784738572523095, 'colsample_bytree': 0.7845045193801675, 'min_child_samples': 5, 'num_leaves': 28, 'feature_fraction': 0.6850891104321319, 'bagging_fraction': 0.7374090593525049}. Best is trial 0 with value: 0.8705873402312843.
[I 2025-01-16 03:52:53,772] Trial 1 finished with value: 0.8814589665653495 and parameters: {'n_estimators': 348, 'max_depth': 65, 'learning_rate': 0.15275538581127557, 'subsample': 0.5134430310390805, 'colsample_bytree': 0.7318824193992858, 'min_child_samples': 6, 'num_leaves': 27, 'feature_fraction': 0.6509501613267878, 'bagging_fraction': 0.7472805132964118}. Best is trial 1 with value: 0.8814589665653495.
[I 2025-01-16 03:52:54,536] Trial 2 finished with value: 0.8

Best parameters for LightGBM: {'n_estimators': 353, 'max_depth': 66, 'learning_rate': 0.16737399597260014, 'subsample': 0.5695120914593372, 'colsample_bytree': 0.7398636753685246, 'min_child_samples': 5, 'num_leaves': 28, 'feature_fraction': 0.6846680225456911, 'bagging_fraction': 0.7920492707864726}
Best Macro F1 Score for LightGBM: 0.9131901718731393


In [ ]:
# 데이터 분리
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42,shuffle=True)

# SMOTE 적용
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

def optimize_catboost(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 765, 775),
        'depth': trial.suggest_int('depth', 3, 5),
        'learning_rate': trial.suggest_float('learning_rate', 0.09, 0.1),
        'subsample': trial.suggest_float('subsample', 0.9, 1.0),
        'random_state': 42,  # 고정된 랜덤 시드
        'verbose': 0,  # 출력 제어
        #'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10),
        'colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.9, 1.0)
    }
    model = CatBoostClassifier(**params)
    model.fit(X_train_smote, y_train_smote)
    y_pred = model.predict(X_test)
    return f1_score(y_test, y_pred, average='macro')

study_cat = optuna.create_study(direction='maximize')
study_cat.optimize(optimize_catboost, n_trials=100)
print("Best parameters for CatBoost:", study_cat.best_params)
print("Best Macro F1 Score for CatBoost:", study_cat.best_value)


[I 2025-01-16 03:58:23,693] A new study created in memory with name: no-name-f69ee32f-8520-4994-81bb-ef88aaf351c2
[I 2025-01-16 03:58:25,988] Trial 0 finished with value: 0.8746435925867257 and parameters: {'iterations': 768, 'depth': 3, 'learning_rate': 0.0933848600933555, 'subsample': 0.904432810564826, 'colsample_bylevel': 0.933570482160207}. Best is trial 0 with value: 0.8746435925867257.
[I 2025-01-16 03:58:30,602] Trial 1 finished with value: 0.8825657101519171 and parameters: {'iterations': 768, 'depth': 5, 'learning_rate': 0.09891350063805138, 'subsample': 0.9067843824491937, 'colsample_bylevel': 0.9741165079895057}. Best is trial 1 with value: 0.8825657101519171.
[I 2025-01-16 03:58:35,661] Trial 2 finished with value: 0.8866123399301513 and parameters: {'iterations': 769, 'depth': 5, 'learning_rate': 0.0979418361994029, 'subsample': 0.9969832395970732, 'colsample_bylevel': 0.9500418351124723}. Best is trial 2 with value: 0.8866123399301513.
[I 2025-01-16 03:58:38,167] Trial 3

Best parameters for CatBoost: {'iterations': 770, 'depth': 3, 'learning_rate': 0.09545073710689564, 'subsample': 0.9956197806930602, 'colsample_bylevel': 0.9579137038910763}
Best Macro F1 Score for CatBoost: 0.9006527947776417


**K-FOLD 10 검증**

In [47]:
base_models = [
    ('xgb', XGBClassifier(
        eval_metric='logloss', random_state=42,
        n_estimators=832, max_depth=12, learning_rate=0.20593358762047578,
        subsample=0.5075069759848869, colsample_bytree=0.7424805892464985,
        min_child_weight=5.409454498304978, gamma=0.9114298633986331
    )),
    ('cat', CatBoostClassifier(
        verbose=0, random_state=42,
        iterations=770, depth=3, learning_rate=0.09545073710689564,
        subsample=0.9956197806930602, colsample_bylevel=0.9579137038910763
    )),
    ('lgb', LGBMClassifier(
        random_state=42,
        n_estimators=353, max_depth=66, learning_rate=0.16737399597260014,
        subsample=0.5695120914593372, colsample_bytree=0.7398636753685246,
        min_child_samples=5, num_leaves=28, feature_fraction=0.6846680225456911,
        bagging_fraction=0.7920492707864726, verbose=-1
    ))
]

voting_clf = VotingClassifier(estimators=base_models, voting='soft')

kf = KFold(n_splits=10, shuffle=True, random_state=SEED)

scores = {
    "macro_f1": [],
    "accuracy": [],
    "precision": [],
    "recall": []
}


for train_index, test_index in kf.split(X):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    voting_clf.fit(X_train, y_train)

    y_pred = voting_clf.predict(X_test)

    scores["macro_f1"].append(f1_score(y_test, y_pred, average='macro'))
    scores["accuracy"].append(accuracy_score(y_test, y_pred))
    scores["precision"].append(precision_score(y_test, y_pred, average='macro'))
    scores["recall"].append(recall_score(y_test, y_pred, average='macro'))

print("Average Macro F1 Score:", np.mean(scores["macro_f1"]))
print("Average Accuracy:", np.mean(scores["accuracy"]))
print("Average Macro Precision:", np.mean(scores["precision"]))
print("Average Macro Recall:", np.mean(scores["recall"]))

Average Macro F1 Score: 0.9196654091730876
Average Accuracy: 0.9675554880928287
Average Macro Precision: 0.9428049119126026
Average Macro Recall: 0.9031332897258721


**모델 훈련**

In [48]:
# 베이스 모델
base_models = [
    ('xgb', XGBClassifier(
        eval_metric='logloss', random_state=42,
        n_estimators=832, max_depth=12, learning_rate=0.20593358762047578,
        subsample=0.5075069759848869, colsample_bytree=0.7424805892464985,
        min_child_weight=5.409454498304978, gamma=0.9114298633986331
    )),
    ('cat', CatBoostClassifier(
        verbose=0, random_state=42,
        iterations=770, depth=3, learning_rate=0.09545073710689564,
        subsample=0.9956197806930602, colsample_bylevel=0.9579137038910763
    )),
    ('lgb', LGBMClassifier(
        random_state=42,
        n_estimators=353, max_depth=66, learning_rate=0.16737399597260014,
        subsample=0.5695120914593372, colsample_bytree=0.7398636753685246,
        min_child_samples=5, num_leaves=28, feature_fraction=0.6846680225456911,
        bagging_fraction=0.7920492707864726, verbose=-1
    ))
]

voting_clf = VotingClassifier(estimators=base_models, voting='soft')

    # 보팅 모델 학습
voting_clf.fit(X, y)

VotingClassifier(estimators=[('xgb',
                              XGBClassifier(base_score=None, booster=None,
                                            callbacks=None,
                                            colsample_bylevel=None,
                                            colsample_bynode=None,
                                            colsample_bytree=0.7424805892464985,
                                            device=None,
                                            early_stopping_rounds=None,
                                            enable_categorical=False,
                                            eval_metric='logloss',
                                            feature_types=None,
                                            gamma=0.9114298633986331,
                                            grow_policy=None,
                                            importance_type=None,
                                            interac...
                              <catboost.core.CatBoostClassifier object at 0x7f36811e1950>),
                             ('lgb',
                              LGBMClassifier(bagging_fraction=0.7920492707864726,
                                             colsample_bytree=0.7398636753685246,
                                             feature_fraction=0.6846680225456911,
                                             learning_rate=0.16737399597260014,
                                             max_depth=66, min_child_samples=5,
                                             n_estimators=353, num_leaves=28,
                                             random_state=42,
                                             subsample=0.5695120914593372,
                                             verbose=-1))],
                 voting='soft')

**제출**

In [54]:
test1 = test

In [55]:
pred = pd.Series(voting_clf.predict(test1))

In [56]:
submit = pd.read_csv('./sample_submission.csv')


In [57]:
submit['허위매물여부'] = pred
submit.head()
submit.to_csv('./final_submission.csv',index=False)